# 06 · Collecte de nouveaux produits via une API

**Ce que fait ce notebook.** La marketplace envisage d'élargir sa gamme à l'épicerie fine. Avant tout
modèle, une question plus terre à terre : peut-on récupérer automatiquement ces produits, avec les
informations nécessaires ?

**Ce qu'il établit.** La collecte fonctionne, mais dix produits suffisent à révéler un enjeu de
qualité et d'homogénéité des métadonnées, à traiter en amont du modèle.

In [1]:
import sys

sys.path.insert(0, "..")
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

## Source et correspondance des champs

Open Food Facts ne demande aucune inscription : le notebook reste exécutable par un tiers, sans clé à
transmettre. Cette base est alimentée de façon collaborative, ce qui aura son importance.

Les cinq champs attendus viennent du schéma d'Edamam, l'autre source proposée. Quatre
correspondances sont évidentes ; la cinquième demande un jugement : `foodContentsLabel` désigne la
composition d'un produit, dont `ingredients_text` est l'équivalent le plus proche.

In [2]:
from collecte_api import CORRESPONDANCE, interroger, normaliser

pd.DataFrame([{"Champ demandé": k, "Champ Open Food Facts": v} for k, v in CORRESPONDANCE.items()])

,Champ demandé,Champ Open Food Facts
0,foodId,code
1,label,product_name
2,category,categories
3,foodContentsLabel,ingredients_text
4,image,image_url


## Collecte

Le filtre porte sur la catégorie et non sur le texte libre : une recherche plein texte remonterait
aussi tout ce qui mentionne le mot sans en être : vinaigres, sauces, arômes.

In [3]:
produits = [normaliser(p) for p in interroger("champagne", 10)]
collecte = pd.DataFrame(produits)

print(f"{len(collecte)} produits collectés")
for champ in CORRESPONDANCE:
    print(f"  {champ:20s} renseigné pour {(collecte[champ] != '').sum()}/{len(collecte)}")

10 produits collectés
  foodId               renseigné pour 10/10
  label                renseigné pour 10/10
  category             renseigné pour 10/10
  foodContentsLabel    renseigné pour 8/10
  image                renseigné pour 10/10


In [4]:
collecte[["foodId", "label", "category"]]

,foodId,label,category
0,3282946015837,Nicolas Feuillatte,"French Champagnes, fr:Champagnes bruts"
1,3185370729960,Br МОЁ HANDON MOET & CHANDON CHAMPAGNE IMPERIA...,Champagnes
2,3049614222245,Champagne Veuve Clicquot brut,Champagnes
3,3043700103715,Champagne brut Cordon Rouge,"French Champagnes, fr:Champagnes bruts"
4,3282946100090,Champagne brut rosé,fr:Champagnes rosés
5,3049610004104,Veuve Clicquot Champagne Ponsardin Brut,Champagnes
6,8000570083306,"MARTINI Bellini Peach 8,0%vol",Champagnes
7,3113934004147,Canard Duchêne,"Champagnes, fr:Liquide"
8,3359952005005,"Champagne AOP, brut",fr:Champagnes bruts
9,3267851000116,Champagne Orgueil de France,French Champagnes


## Constats

Trois observations, sur dix produits seulement.

**Les catégories sont hétérogènes.** Certaines étiquettes n'ont pas été traduites et conservent un
préfixe de langue ; l'une d'elles ne dit à peu près rien du produit.

In [5]:
from collections import Counter

etiquettes = Counter(c.strip() for ligne in collecte["category"] for c in ligne.split(","))
pd.Series(etiquettes).sort_values(ascending=False).to_frame("produits concernés")

,produits concernés
Champagnes,5
French Champagnes,3
fr:Champagnes bruts,3
fr:Champagnes rosés,1
fr:Liquide,1


**Les libellés sont irréguliers**, trace visible de la saisie collaborative et de la reconnaissance
automatique d'étiquettes. **Et la catégorie source n'est pas toujours juste** : un cocktail à la
pêche figure parmi les champagnes.

In [6]:
for ligne in collecte.itertuples():
    contenu = ligne.foodContentsLabel[:60] if ligne.foodContentsLabel else "— vide —"
    print(f"  {ligne.label[:52]:<52} | {contenu}")

  Nicolas Feuillatte                                   | Champagne, Contient des _sulfites_
  Br МОЁ HANDON MOET & CHANDON CHAMPAGNE IMPERIAL BR   | IMPERIAL LOOK BEHIND THE SCENES OF OUR MAISON SERIE CALL CON
  Champagne Veuve Clicquot brut                        | — vide —
  Champagne brut Cordon Rouge                          | Contient des sulfites.
  Champagne brut rosé                                  | — vide —
  Veuve Clicquot Champagne Ponsardin Brut              | Champagne
  MARTINI Bellini Peach 8,0%vol                        | Wein (Sulfite), Wasser, Zucker, Aromastoffe, Kohlendioxid, S
  Canard Duchêne                                       | Pinots et de Chardonnay
  Champagne AOP, brut                                  | Champagne brut (_sulfites_)
  Champagne Orgueil de France                          | Champagne


In [7]:
from pathlib import Path

sortie = Path("..") / "reports" / "produits_champagne.csv"
collecte.to_csv(sortie, index=False)
print(f"écrit dans {sortie}")

écrit dans ../reports/produits_champagne.csv


**Ce que ce notebook établit.** La collecte automatique fonctionne et produit le fichier demandé. Elle
reste exploratoire, dix produits ne caractérisant pas une base entière, mais elle met déjà en
évidence un enjeu de qualité et d'homogénéité des métadonnées, qui devra être traité en amont du
modèle de classification. On retrouve, sous une autre forme, la difficulté du point de départ : ici
comme sur la marketplace, les catégories sont déclarées par des contributeurs qui ne suivent pas
tous la même règle.